# 02 — Ingesta INPC + SHCP + IGAE + ENOE → Parquet (Bronce)

**Proyecto:** Mexico Economic Pipeline  
**Fuentes:** INEGI (INPC · IGAE) · SHCP (Finanzas Públicas)  
**Output:** `data/raw/inpc/parquet/` · `data/raw/shcp/parquet/` · `data/raw/igae/parquet/`

Este notebook lee los tres archivos fuente, aplica transformaciones mínimas de estructura
y los persiste como Parquet en la capa Bronce. La limpieza de contenido va en `06_transformacion.ipynb`.

---
### Estructura esperada de archivos fuente
```
data/raw/
├── inpc/
│   └── conjunto_de_datos_inpc_mensual.csv   ← BIE INEGI, formato long, 7 columnas
├── shcp/
│   ├── ingreso_gasto_finan.csv              ← SHCP Estadísticas Oportunas, reciente
│   └── ingreso_gasto_finan_hist.csv         ← histórico
└── igae/
    └── IGAE_2.xlsx                          ← INEGI, hoja "Tabulado", formato wide
```

### Formato INPC (BIE INEGI)
```
COBERTURA | PERIODICIDAD | FECHA       | CONCEPTO                        | VALOR   | UNIDAD_MEDIDA | ESTATUS
Nacional  | Mensual      | 2021-02-01  | ...Precios al Consumidor (INPC)  | 110.907 | Índice        | Cifras definitivas
```
~1,989 filas · 7 subíndices · cobertura: feb 2021 – abr 2026

### Formato SHCP (Finanzas Públicas)
16 columnas · `CICLO` (año), `MES` (nombre en español), `CLAVE_DE_CONCEPTO`, `MONTO` (miles de pesos)  
XBB = Ingresos · XBC = Gasto · Sector: Gobierno Federal Presupuestario · 1990–2026

### Formato IGAE (INEGI Excel)
Hoja `Tabulado`, formato wide: actividades como filas, periodos mensuales como columnas.  
4 series: Total · Actividades primarias · secundarias · terciarias · 1993–2026 · base 2018=100

## 0 · Imports y configuración de Spark

In [1]:
import sys, os
from pathlib import Path
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, LongType, DoubleType
)

spark = (
    SparkSession.builder
    .appName("ingesta_inpc_shcp_igae")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.parquet.compression.codec", "snappy")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} iniciado")

Spark 3.5.9 iniciado


In [2]:
# ── Paths base (relativos a la raíz del repo) ────────────────────────────────
BASE  = Path("..")
RAW   = BASE / "data" / "raw"

# Fuentes INPC
INPC_CSV = str(RAW / "inpc" / "conjunto_de_datos_inpc_mensual.csv")

# Fuentes SHCP (ingreso/gasto gobierno federal)
SHCP_CSV      = str(RAW / "shcp" / "ingreso_gasto_finan.csv")
SHCP_HIST_CSV = str(RAW / "shcp" / "ingreso_gasto_finan_hist.csv")

# Fuente IGAE (Excel INEGI)
IGAE_XLSX = str(RAW / "igae" / "IGAE_2.xlsx")

# Destinos Parquet (bronce)
INPC_PARQUET = str(RAW / "inpc" / "parquet")
SHCP_PARQUET = str(RAW / "shcp" / "parquet")
IGAE_PARQUET = str(RAW / "igae" / "parquet")

# Crear directorios de salida
for d in [INPC_PARQUET, SHCP_PARQUET, IGAE_PARQUET]:
    Path(d).mkdir(parents=True, exist_ok=True)

print("Paths configurados ✓")
print(f"  INPC fuente    : {INPC_CSV}")
print(f"  SHCP reciente  : {SHCP_CSV}")
print(f"  SHCP histórico : {SHCP_HIST_CSV}")
print(f"  IGAE Excel     : {IGAE_XLSX}")
print(f"  INPC parquet   : {INPC_PARQUET}")
print(f"  SHCP parquet   : {SHCP_PARQUET}")
print(f"  IGAE parquet   : {IGAE_PARQUET}")

ENOE_DIR     = str(RAW / "enoe" / "zips")      # Carpeta con los 85 ZIPs
ENOE_RAW_DIR = str(RAW / "enoe" / "raw")       # Destino de descompresión
ENOE_PARQUET = str(RAW / "enoe" / "parquet")   # Parquet bronce

# Crear directorio de descompresión si no existe
Path(ENOE_RAW_DIR).mkdir(parents=True, exist_ok=True)

Paths configurados ✓
  INPC fuente    : ..\data\raw\inpc\conjunto_de_datos_inpc_mensual.csv
  SHCP reciente  : ..\data\raw\shcp\ingreso_gasto_finan.csv
  SHCP histórico : ..\data\raw\shcp\ingreso_gasto_finan_hist.csv
  IGAE Excel     : ..\data\raw\igae\IGAE_2.xlsx
  INPC parquet   : ..\data\raw\inpc\parquet
  SHCP parquet   : ..\data\raw\shcp\parquet
  IGAE parquet   : ..\data\raw\igae\parquet


---
## 1 · INPC — Índice Nacional de Precios al Consumidor

El archivo `conjunto_de_datos_inpc_mensual.csv` del BIE ya viene en formato **long**.
Columnas: `COBERTURA`, `PERIODICIDAD`, `FECHA`, `CONCEPTO`, `VALOR`, `UNIDAD_MEDIDA`, `ESTATUS`.

Pasos:
1. Leer el CSV tal cual
2. Renombrar columnas a snake_case
3. Extraer el nombre corto del subíndice desde `CONCEPTO` (última parte de la jerarquía)
4. Derivar `anio`, `mes`, `trimestre` desde `FECHA`
5. Guardar Parquet particionado por `anio`

In [3]:
# ── 1.1 Leer INPC crudo ──────────────────────────────────────────────────────
df_inpc_raw = (
    spark.read
    .option("header", "true")
    .option("encoding", "UTF-8")
    .option("inferSchema", "false")
    .csv(INPC_CSV)
)

print(f"Filas    : {df_inpc_raw.count():,}")
print(f"Columnas : {df_inpc_raw.columns}")
df_inpc_raw.show(5, truncate=80)

Filas    : 1,988
Columnas : ['COBERTURA', 'PERIODICIDAD', 'FECHA', 'CONCEPTO', 'VALOR', 'UNIDAD_MEDIDA', 'ESTATUS']
+---------+------------+----------+--------------------------------------------------------------------------------+-------+-------------+------------------+
|COBERTURA|PERIODICIDAD|     FECHA|                                                                        CONCEPTO|  VALOR|UNIDAD_MEDIDA|           ESTATUS|
+---------+------------+----------+--------------------------------------------------------------------------------+-------+-------------+------------------+
| Nacional|     Mensual|2021-02-01|Índice nacional de precios al consumidor (mensual), Resumen, Subíndices subya...|110.907|       Índice|Cifras definitivas|
| Nacional|     Mensual|2021-02-01|Índice nacional de precios al consumidor (mensual), Resumen, Subíndices subya...|110.061|       Índice|Cifras definitivas|
| Nacional|     Mensual|2021-02-01|Índice nacional de precios al consumidor (mensual), Resumen

In [4]:
# ── 1.2 Renombrar columnas a snake_case y seleccionar las útiles ─────────────
# Columnas originales: COBERTURA, PERIODICIDAD, FECHA, CONCEPTO, VALOR, UNIDAD_MEDIDA, ESTATUS
# Nos quedamos con: FECHA, CONCEPTO, VALOR  (el resto son constantes o metadatos)

df_inpc = (
    df_inpc_raw
    .withColumnRenamed("FECHA",    "fecha")
    .withColumnRenamed("CONCEPTO", "concepto")
    .withColumnRenamed("VALOR",    "valor")
    .select("fecha", "concepto", "valor")
)

# ── 1.3 Extraer nombre corto del subíndice desde CONCEPTO ────────────────────
# CONCEPTO es una jerarquía separada por comas:
#   "Índice nacional..., Resumen, Subíndices..., Precios al Consumidor (INPC)"
#   "Índice nacional..., Resumen, Subíndices..., Subyacente"
#   "Índice nacional..., Resumen, Subíndices..., Subyacente, Mercancías"
# → tomamos la ÚLTIMA parte después de la última coma como nombre corto

df_inpc = df_inpc.withColumn(
    "subindice",
    F.trim(F.element_at(F.split(F.col("concepto"), ","), -1))
)

print("Subíndices disponibles:")
df_inpc.select("subindice").distinct().orderBy("subindice").show(truncate=False)

Subíndices disponibles:
+-------------------------------------------------+
|subindice                                        |
+-------------------------------------------------+
|Agropecuarios                                    |
|Energéticos y tarifas autorizadas por el gobierno|
|Mercancías                                       |
|No subyacente                                    |
|Precios al Consumidor (INPC)                     |
|Servicios                                        |
|Subyacente                                       |
+-------------------------------------------------+



In [5]:
# ── 1.4 Derivar anio, mes y trimestre desde FECHA (formato YYYY-MM-DD) ────────
df_inpc = (
    df_inpc
    .withColumn("anio",      F.year(F.to_date(F.col("fecha"), "yyyy-MM-dd")).cast(IntegerType()))
    .withColumn("mes",       F.month(F.to_date(F.col("fecha"), "yyyy-MM-dd")).cast(IntegerType()))
    .withColumn("trimestre", F.ceil(F.col("mes") / 3).cast(IntegerType()))
    .withColumn("indice_valor", F.col("valor").cast(DoubleType()))
    .drop("valor", "concepto")   # concepto ya está resumida en subindice
)

print("Schema final:")
df_inpc.printSchema()

print("\nMuestra:")
df_inpc.show(8, truncate=False)

print("\nRango temporal:")
df_inpc.select(F.min("anio"), F.max("anio"), F.countDistinct("subindice").alias("n_subindices")).show()

Schema final:
root
 |-- fecha: string (nullable = true)
 |-- subindice: string (nullable = true)
 |-- anio: integer (nullable = true)
 |-- mes: integer (nullable = true)
 |-- trimestre: integer (nullable = true)
 |-- indice_valor: double (nullable = true)


Muestra:
+----------+-------------------------------------------------+----+---+---------+------------+
|fecha     |subindice                                        |anio|mes|trimestre|indice_valor|
+----------+-------------------------------------------------+----+---+---------+------------+
|2021-02-01|Precios al Consumidor (INPC)                     |2021|2  |1        |110.907     |
|2021-02-01|Subyacente                                       |2021|2  |1        |110.061     |
|2021-02-01|Mercancías                                       |2021|2  |1        |112.216     |
|2021-02-01|Servicios                                        |2021|2  |1        |107.734     |
|2021-02-01|No subyacente                                    |2021|2

In [6]:
# ── 1.5 Guardar como Parquet bronce ──────────────────────────────────────────
(
    df_inpc
    .write
    .mode("overwrite")
    .partitionBy("anio")
    .parquet(INPC_PARQUET)
)

print(f"✅ INPC guardado en: {INPC_PARQUET}")

# Verificar lectura
df_check = spark.read.parquet(INPC_PARQUET)
print(f"   Filas totales  : {df_check.count():,}")
print(f"   Columnas       : {df_check.columns}")
df_check.groupBy("anio").count().orderBy("anio").show()

✅ INPC guardado en: ..\data\raw\inpc\parquet
   Filas totales  : 1,988
   Columnas       : ['fecha', 'subindice', 'mes', 'trimestre', 'indice_valor', 'anio']
+----+-----+
|anio|count|
+----+-----+
|2003|   84|
|2004|   84|
|2005|   84|
|2006|   84|
|2007|   84|
|2008|   84|
|2009|   84|
|2010|   84|
|2011|   84|
|2012|   84|
|2013|   84|
|2014|   84|
|2015|   84|
|2016|   84|
|2017|   84|
|2018|   84|
|2019|   84|
|2020|   84|
|2021|   84|
|2022|   84|
+----+-----+
only showing top 20 rows



---
## 2 · SHCP — Ingresos y Gastos del Gobierno Federal

Fuente: [SHCP · Estadísticas Oportunas de Finanzas Públicas](https://www.finanzaspublicas.hacienda.gob.mx/)

Dos archivos CSV:
- `ingreso_gasto_finan.csv` — datos recientes
- `ingreso_gasto_finan_hist.csv` — datos históricos

Schema (16 columnas): `CICLO, MES, CLAVE_DE_CONCEPTO, NOMBRE, TEMA, SUBTEMA, SECTOR, AMBITO,
TIPO_DE_INFORMACION, BASE_DE_REGISTRO, UNIDAD_DE_MEDIDA, PERIODO_INICIO, PERIODO_FINAL, FRECUENCIA, DIFUSION, MONTO`

**Series de interés:**
- `XBB` — Ingresos del Gobierno Federal
- `XBC` — Gasto Total del Gobierno Federal
- Sector: `Gobierno Federal Presupuestario`
- Cobertura: 1990–2026 · ~878 registros limpios

In [7]:
# ── 2.1 Leer ambos CSV y unirlos ──────────────────────────────────────────────
df_shcp_raw1 = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("encoding", "UTF-8")
    .csv(SHCP_CSV)
)

df_shcp_raw2 = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("encoding", "UTF-8")
    .csv(SHCP_HIST_CSV)
)

df_shcp_crudo = df_shcp_raw1.unionByName(df_shcp_raw2, allowMissingColumns=True)

print(f"Reciente  : {df_shcp_raw1.count():,} filas")
print(f"Histórico : {df_shcp_raw2.count():,} filas")
print(f"Total     : {df_shcp_crudo.count():,} filas")
print(f"Columnas  : {df_shcp_crudo.columns}")
df_shcp_crudo.show(3, truncate=60)

Reciente  : 863,376 filas
Histórico : 578,337 filas
Total     : 1,441,713 filas
Columnas  : ['CICLO', 'MES', 'CLAVE_DE_CONCEPTO', 'NOMBRE', 'TEMA', 'SUBTEMA', 'SECTOR', 'AMBITO', 'TIPO_DE_INFORMACION', 'BASE_DE_REGISTRO', 'UNIDAD_DE_MEDIDA', 'PERIODO_INICIO', 'PERIODO_FINAL', 'FRECUENCIA', 'DIFUSION', 'MONTO']
+-----+-------+-----------------+----------------------------------------+-----------------------------------------+-------+-------------------------------------+-------+-------------------+----------------+----------------+-------------------+-------------------+----------+------------------------------------------------+-------------+
|CICLO|    MES|CLAVE_DE_CONCEPTO|                                  NOMBRE|                                     TEMA|SUBTEMA|                               SECTOR| AMBITO|TIPO_DE_INFORMACION|BASE_DE_REGISTRO|UNIDAD_DE_MEDIDA|     PERIODO_INICIO|      PERIODO_FINAL|FRECUENCIA|                                        DIFUSION|        MONTO|
+-----+---

In [8]:
# ── 2.2 Filtrar series XBB (Ingresos) y XBC (Gasto) del Gobierno Federal ──────
series_filtradas = df_shcp_crudo.filter(
    F.col("CLAVE_DE_CONCEPTO").isin("XBB", "XBC") &
    (F.col("SECTOR") == "Gobierno Federal Presupuestario")
)

print(f"Filas filtradas : {series_filtradas.count():,}")
series_filtradas.select("CLAVE_DE_CONCEPTO", "NOMBRE", "SECTOR").distinct().show(truncate=False)

Filas filtradas : 1,180
+-----------------+--------------------------------+-------------------------------+
|CLAVE_DE_CONCEPTO|NOMBRE                          |SECTOR                         |
+-----------------+--------------------------------+-------------------------------+
|XBB              |Ingresos del Gobierno Federal   |Gobierno Federal Presupuestario|
|XBC              |Gasto Total del Gobierno Federal|Gobierno Federal Presupuestario|
+-----------------+--------------------------------+-------------------------------+



In [9]:
# ── 2.3 Eliminar duplicados ────────────────────────────────────────────────────
n_antes = series_filtradas.count()
series_limpias = series_filtradas.dropDuplicates()
n_despues = series_limpias.count()

print(f"Duplicados eliminados: {n_antes - n_despues:,}  ({n_antes:,} → {n_despues:,})")
# Esperado: ~1180 → 878

Duplicados eliminados: 302  (1,180 → 878)


In [10]:
# ── 2.4 Convertir MES (texto en español) → MES_NUM (entero) ──────────────────
mes_num = (
    F.when(F.col("MES") == "Enero",      1)
     .when(F.col("MES") == "Febrero",    2)
     .when(F.col("MES") == "Marzo",      3)
     .when(F.col("MES") == "Abril",      4)
     .when(F.col("MES") == "Mayo",       5)
     .when(F.col("MES") == "Junio",      6)
     .when(F.col("MES") == "Julio",      7)
     .when(F.col("MES") == "Agosto",     8)
     .when(F.col("MES") == "Septiembre", 9)
     .when(F.col("MES") == "Octubre",    10)
     .when(F.col("MES") == "Noviembre",  11)
     .when(F.col("MES") == "Diciembre",  12)
)

series_limpias = series_limpias.withColumn("MES_NUM", mes_num)

print("Verificación MES → MES_NUM:")
series_limpias.select("MES", "MES_NUM").distinct().orderBy("MES_NUM").show()

Verificación MES → MES_NUM:
+----------+-------+
|       MES|MES_NUM|
+----------+-------+
|     Enero|      1|
|   Febrero|      2|
|     Marzo|      3|
|     Abril|      4|
|      Mayo|      5|
|     Junio|      6|
|     Julio|      7|
|    Agosto|      8|
|Septiembre|      9|
|   Octubre|     10|
| Noviembre|     11|
| Diciembre|     12|
+----------+-------+



In [11]:
# ── 2.5 Construir columna FECHA y trimestre ───────────────────────────────────
series_limpias = (
    series_limpias
    .withColumn("FECHA",     F.make_date(F.col("CICLO"), F.col("MES_NUM"), F.lit(1)))
    .withColumn("TRIMESTRE", F.ceil(F.col("MES_NUM") / 3).cast(IntegerType()))
)

print("Muestra con FECHA y TRIMESTRE:")
series_limpias.select("CICLO", "MES", "MES_NUM", "FECHA", "TRIMESTRE").show(5)

print("\nRango temporal:")
series_limpias.select(
    F.min("FECHA").alias("primera"),
    F.max("FECHA").alias("ultima"),
    F.countDistinct("FECHA").alias("periodos")
).show()

Muestra con FECHA y TRIMESTRE:
+-----+---------+-------+----------+---------+
|CICLO|      MES|MES_NUM|     FECHA|TRIMESTRE|
+-----+---------+-------+----------+---------+
| 2011|   Agosto|      8|2011-08-01|        3|
| 2011|Diciembre|     12|2011-12-01|        4|
| 2011|    Junio|      6|2011-06-01|        2|
| 2012|    Junio|      6|2012-06-01|        2|
| 2012|   Agosto|      8|2012-08-01|        3|
+-----+---------+-------+----------+---------+
only showing top 5 rows


Rango temporal:
+----------+----------+--------+
|   primera|    ultima|periodos|
+----------+----------+--------+
|1990-01-01|2026-07-01|     439|
+----------+----------+--------+



In [12]:
# ── 2.6 Seleccionar y ordenar columnas finales ────────────────────────────────
# Columnas de salida (alineadas con Lab 5):
# FECHA, CICLO, MES, MES_NUM, TRIMESTRE, CLAVE_DE_CONCEPTO, NOMBRE,
# SUBTEMA, SECTOR, UNIDAD_DE_MEDIDA, MONTO

df_shcp = series_limpias.select(
    "FECHA",
    "CICLO",
    "MES",
    "MES_NUM",
    "TRIMESTRE",
    "CLAVE_DE_CONCEPTO",
    "NOMBRE",
    "SUBTEMA",
    "SECTOR",
    "UNIDAD_DE_MEDIDA",
    "MONTO"
).orderBy("FECHA", "CLAVE_DE_CONCEPTO")

print("Schema final SHCP:")
df_shcp.printSchema()

print(f"\nTotal filas: {df_shcp.count():,}")
df_shcp.show(10, truncate=50)

Schema final SHCP:
root
 |-- FECHA: date (nullable = true)
 |-- CICLO: integer (nullable = true)
 |-- MES: string (nullable = true)
 |-- MES_NUM: integer (nullable = true)
 |-- TRIMESTRE: integer (nullable = true)
 |-- CLAVE_DE_CONCEPTO: string (nullable = true)
 |-- NOMBRE: string (nullable = true)
 |-- SUBTEMA: string (nullable = true)
 |-- SECTOR: string (nullable = true)
 |-- UNIDAD_DE_MEDIDA: string (nullable = true)
 |-- MONTO: double (nullable = true)


Total filas: 878
+----------+-----+-------+-------+---------+-----------------+--------------------------------+-------+-------------------------------+----------------+-----------+
|     FECHA|CICLO|    MES|MES_NUM|TRIMESTRE|CLAVE_DE_CONCEPTO|                          NOMBRE|SUBTEMA|                         SECTOR|UNIDAD_DE_MEDIDA|      MONTO|
+----------+-----+-------+-------+---------+-----------------+--------------------------------+-------+-------------------------------+----------------+-----------+
|1990-01-01| 1990|  Ene

In [13]:
# ── 2.7 Verificar cobertura por serie ─────────────────────────────────────────
print("Registros por serie:")
df_shcp.groupBy("CLAVE_DE_CONCEPTO", "NOMBRE") \
       .agg(
           F.count("*").alias("n"),
           F.min("FECHA").alias("desde"),
           F.max("FECHA").alias("hasta")
       ).show(truncate=50)

Registros por serie:
+-----------------+--------------------------------+---+----------+----------+
|CLAVE_DE_CONCEPTO|                          NOMBRE|  n|     desde|     hasta|
+-----------------+--------------------------------+---+----------+----------+
|              XBB|   Ingresos del Gobierno Federal|439|1990-01-01|2026-07-01|
|              XBC|Gasto Total del Gobierno Federal|439|1990-01-01|2026-07-01|
+-----------------+--------------------------------+---+----------+----------+



In [14]:
# ── 2.8 Guardar como Parquet bronce particionado por CICLO ───────────────────
(
    df_shcp
    .write
    .mode("overwrite")
    .partitionBy("CICLO")
    .parquet(SHCP_PARQUET)
)

print(f"✅ SHCP guardado en: {SHCP_PARQUET}")

# Verificar lectura
df_check = spark.read.parquet(SHCP_PARQUET)
print(f"   Filas totales  : {df_check.count():,}")
print(f"   Columnas       : {df_check.columns}")
df_check.groupBy("CICLO").count().orderBy("CICLO").show(40)

✅ SHCP guardado en: ..\data\raw\shcp\parquet
   Filas totales  : 878
   Columnas       : ['FECHA', 'MES', 'MES_NUM', 'TRIMESTRE', 'CLAVE_DE_CONCEPTO', 'NOMBRE', 'SUBTEMA', 'SECTOR', 'UNIDAD_DE_MEDIDA', 'MONTO', 'CICLO']
+-----+-----+
|CICLO|count|
+-----+-----+
| 1990|   24|
| 1991|   24|
| 1992|   24|
| 1993|   24|
| 1994|   24|
| 1995|   24|
| 1996|   24|
| 1997|   24|
| 1998|   24|
| 1999|   24|
| 2000|   24|
| 2001|   24|
| 2002|   24|
| 2003|   24|
| 2004|   24|
| 2005|   24|
| 2006|   24|
| 2007|   24|
| 2008|   24|
| 2009|   24|
| 2010|   24|
| 2011|   24|
| 2012|   24|
| 2013|   24|
| 2014|   24|
| 2015|   24|
| 2016|   24|
| 2017|   24|
| 2018|   24|
| 2019|   24|
| 2020|   24|
| 2021|   24|
| 2022|   24|
| 2023|   24|
| 2024|   24|
| 2025|   24|
| 2026|   14|
+-----+-----+



---
## 3 · IGAE — Indicador Global de Actividad Económica

Fuente: [INEGI · IGAE](https://www.inegi.org.mx/temas/igae/)  
Archivo: `data/raw/igae/IGAE_2.xlsx`, hoja `Tabulado`  
Formato original: ancho (actividades × períodos mensuales)  
Cobertura: enero 1993 – más reciente  
Unidad: Índice de volumen físico base 2018 = 100  
Series incluidas: Total · Actividades primarias · secundarias · terciarias

In [15]:
# ── 3.1 Leer Excel con pandas (sin header fijo) ───────────────────────────────
# El tabulado del INEGI tiene títulos y notas antes de los datos reales.
# header=None conserva la estructura original para inspeccionarla.

xls = pd.ExcelFile(IGAE_XLSX)
print("Hojas disponibles:", xls.sheet_names)

raw = pd.read_excel(IGAE_XLSX, sheet_name="Tabulado", header=None)
print(f"Dimensiones del tabulado: {raw.shape}")
print("\nPrimeras 8 filas (vista de estructura):")
print(raw.iloc[:8, :6].to_string())

Hojas disponibles: ['Tabulado', 'MetaInfo']
Dimensiones del tabulado: (138, 443)

Primeras 8 filas (vista de estructura):
                                                                                                                                            0          1          2          3          4          5
0                                                                                      Instituto Nacional de Estadística y Geografía (INEGI).        NaN        NaN        NaN        NaN        NaN
1  Sistema de Cuentas Nacionales de México. Indicador Global de la Actividad Económica. Año Base 2018. Serie de enero de 1993 a julio de 2026        NaN        NaN        NaN        NaN        NaN
2                                                                                                       Series Originales/ Índices 2018 = 100        NaN        NaN        NaN        NaN        NaN
3                                                                                         

In [16]:
# ── 3.2 Localizar fila de años y fila de meses ───────────────────────────────
# Estructura real del tabulado INEGI:
#   - Fila con "Concepto" en col 0 y el AÑO (texto, p.ej. "1993") solo en la
#     primera columna de cada año (celdas combinadas → el resto queda NaN).
#   - Fila siguiente: nombres de mes en español ("Enero", "EneroP", "JunioR"...)
#     más una columna "Anual" por año.
col0 = raw[0].fillna("").astype(str).str.strip()
fila_anios = int(col0[col0 == "Concepto"].index[0])

print(f"Fila de años encontrada en índice: {fila_anios}")
print("Primeros 14 valores años :", list(raw.iloc[fila_anios,     :14]))
print("Primeros 14 valores meses:", list(raw.iloc[fila_anios + 1, :14]))

Fila de años encontrada en índice: 4
Primeros 14 valores años : ['Concepto', '1993', nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan]
Primeros 14 valores meses: [nan, 'Enero', 'Febrero', 'Marzo', 'Abril', 'Mayo', 'Junio', 'Julio', 'Agosto', 'Septiembre', 'Octubre', 'Noviembre', 'Diciembre', 'Anual']


In [17]:
# ── 3.3 Construir fechas mensuales desde las dos filas de cabecera ────────────
# Años: se propagan hacia la derecha (ffill) porque solo aparecen en la 1ª columna.
# Meses: nombres en español; se quitan los sufijos P (preliminar) / R (revisada).
# Las columnas "Anual" se descartan (NaT).
fila_meses = fila_anios + 1

MESES = {"enero": 1, "febrero": 2, "marzo": 3, "abril": 4, "mayo": 5, "junio": 6,
         "julio": 7, "agosto": 8, "septiembre": 9, "octubre": 10,
         "noviembre": 11, "diciembre": 12}

def mes_a_num(txt):
    t = str(txt).strip().lower()
    if t.endswith(("p", "r")) and t[:-1] in MESES:   # "enerop" -> "enero"
        t = t[:-1]
    return MESES.get(t)  # "anual", NaN, etc. -> None

anios_raw = raw.iloc[fila_anios, 1:].ffill()
meses_raw = raw.iloc[fila_meses, 1:]

fechas = []
for a, m in zip(anios_raw.values, meses_raw.values):
    mes = mes_a_num(m)
    try:
        fechas.append(pd.Timestamp(year=int(float(a)), month=mes, day=1) if mes else pd.NaT)
    except Exception:
        fechas.append(pd.NaT)

validas = [f for f in fechas if pd.notna(f)]
print(f"Columnas de período construidas: {len(validas)}")
print(f"Primera fecha: {validas[0]}  |  Última fecha con dato: {validas[-1]}")

Columnas de período construidas: 372
Primera fecha: 1993-01-01 00:00:00  |  Última fecha con dato: 2026-12-01 00:00:00


In [18]:
# ── 3.4 Extraer bloque de índice y seleccionar 4 actividades ─────────────────
# El Excel trae varios bloques (índice, variación anual, acumulados...) con las
# mismas actividades. Nos quedamos solo con el PRIMERO: "Índice de volumen físico".
ACTIVIDADES = [
    "Total",
    "Actividades primarias",
    "Actividades secundarias",
    "Actividades terciarias",
]

col_names = ["ACTIVIDAD"] + fechas
datos_raw = raw.iloc[fila_meses + 1:].copy()
datos_raw.columns = col_names

datos_raw["ACTIVIDAD"] = datos_raw["ACTIVIDAD"].fillna("").astype(str).str.strip()

# Cortar en el segundo bloque (primera fila que empieza con "Variación")
inicio_var = datos_raw.index[datos_raw["ACTIVIDAD"].str.startswith("Variación")][0]
bloque_indice = datos_raw.loc[:inicio_var - 1]

igae_wide = bloque_indice[bloque_indice["ACTIVIDAD"].isin(ACTIVIDADES)].copy()

print(f"Filas seleccionadas: {len(igae_wide)}  (esperado: 4)")
print("Actividades:", igae_wide["ACTIVIDAD"].tolist())

Filas seleccionadas: 4  (esperado: 4)
Actividades: ['Total', 'Actividades primarias', 'Actividades secundarias', 'Actividades terciarias']


In [19]:
# ── 3.5 Transformar wide → long con melt() ───────────────────────────────────
# Columnas de período = todas excepto "ACTIVIDAD"
# Solo mantenemos fechas válidas (pd.Timestamp, no NaT)
cols_fecha = [c for c in igae_wide.columns if isinstance(c, pd.Timestamp)]

igae_long = igae_wide.melt(
    id_vars="ACTIVIDAD",
    value_vars=cols_fecha,
    var_name="FECHA",
    value_name="IGAE"
)

igae_long["FECHA"] = pd.to_datetime(igae_long["FECHA"])  # melt la deja como object

# Eliminar filas sin valor publicado (períodos futuros o sin dato)
igae_long = igae_long.dropna(subset=["IGAE"])
igae_long["IGAE"] = pd.to_numeric(igae_long["IGAE"], errors="coerce")
igae_long = igae_long.dropna(subset=["IGAE"])

# Derivar columnas temporales
igae_long["CICLO"]    = igae_long["FECHA"].dt.year.astype("int64")
igae_long["MES_NUM"]  = igae_long["FECHA"].dt.month.astype("int64")
igae_long["TRIMESTRE"] = ((igae_long["MES_NUM"] - 1) // 3 + 1).astype("int64")

igae_long = igae_long[["FECHA", "ACTIVIDAD", "IGAE", "CICLO", "MES_NUM", "TRIMESTRE"]]
igae_long = igae_long.sort_values(["FECHA", "ACTIVIDAD"]).reset_index(drop=True)

print(f"Registros en formato long: {len(igae_long):,}  (esperado: ~1,612)")
print(f"Rango: {igae_long['FECHA'].min()} → {igae_long['FECHA'].max()}")
print(igae_long.head(8).to_string(index=False))

Registros en formato long: 1,468  (esperado: ~1,612)
Rango: 1993-01-01 00:00:00 → 2026-07-01 00:00:00
     FECHA               ACTIVIDAD      IGAE  CICLO  MES_NUM  TRIMESTRE
1993-01-01   Actividades primarias 58.796198   1993        1          1
1993-01-01 Actividades secundarias 63.307782   1993        1          1
1993-01-01  Actividades terciarias 50.775258   1993        1          1
1993-01-01                   Total 55.434736   1993        1          1
1993-02-01   Actividades primarias 60.763816   1993        2          1
1993-02-01 Actividades secundarias 64.889130   1993        2          1
1993-02-01  Actividades terciarias 51.427224   1993        2          1
1993-02-01                   Total 56.456971   1993        2          1


In [20]:
# ── 3.6 Convertir a Spark DataFrame y verificar schema ───────────────────────
df_igae = spark.createDataFrame(igae_long)

print("Schema Spark:")
df_igae.printSchema()

print("\nActividades y conteo:")
df_igae.groupBy("ACTIVIDAD").count().orderBy("ACTIVIDAD").show(truncate=False)

print("\nRango temporal:")
df_igae.select(
    F.min("FECHA").alias("primera"),
    F.max("FECHA").alias("ultima"),
    F.countDistinct("FECHA").alias("periodos"),
    F.count("*").alias("total_filas")
).show()

Schema Spark:
root
 |-- FECHA: timestamp (nullable = true)
 |-- ACTIVIDAD: string (nullable = true)
 |-- IGAE: double (nullable = true)
 |-- CICLO: long (nullable = true)
 |-- MES_NUM: long (nullable = true)
 |-- TRIMESTRE: long (nullable = true)


Actividades y conteo:
+-----------------------+-----+
|ACTIVIDAD              |count|
+-----------------------+-----+
|Actividades primarias  |367  |
|Actividades secundarias|367  |
|Actividades terciarias |367  |
|Total                  |367  |
+-----------------------+-----+


Rango temporal:
+-------------------+-------------------+--------+-----------+
|            primera|             ultima|periodos|total_filas|
+-------------------+-------------------+--------+-----------+
|1993-01-01 00:00:00|2026-07-01 00:00:00|     367|       1468|
+-------------------+-------------------+--------+-----------+



In [21]:
# ── 3.7 Guardar como Parquet bronce particionado por CICLO ───────────────────
(
    df_igae
    .write
    .mode("overwrite")
    .partitionBy("CICLO")
    .parquet(IGAE_PARQUET)
)

print(f"✅ IGAE guardado en: {IGAE_PARQUET}")

# Verificar lectura
df_check = spark.read.parquet(IGAE_PARQUET)
print(f"   Filas totales  : {df_check.count():,}")
print(f"   Columnas       : {df_check.columns}")
df_check.groupBy("CICLO").count().orderBy("CICLO").show(40)

✅ IGAE guardado en: ..\data\raw\igae\parquet
   Filas totales  : 1,468
   Columnas       : ['FECHA', 'ACTIVIDAD', 'IGAE', 'MES_NUM', 'TRIMESTRE', 'CICLO']
+-----+-----+
|CICLO|count|
+-----+-----+
| 1993|   48|
| 1994|   48|
| 1995|   48|
| 1996|   48|
| 1997|   48|
| 1998|   48|
| 1999|   48|
| 2000|   48|
| 2001|   48|
| 2002|   48|
| 2003|   48|
| 2004|   48|
| 2005|   48|
| 2006|   48|
| 2007|   48|
| 2008|   48|
| 2009|   48|
| 2010|   48|
| 2011|   48|
| 2012|   48|
| 2013|   48|
| 2014|   48|
| 2015|   48|
| 2016|   48|
| 2017|   48|
| 2018|   48|
| 2019|   48|
| 2020|   48|
| 2021|   48|
| 2022|   48|
| 2026|   28|
+-----+-----+



---
## 4 · ENOE — Encuesta Nacional de Ocupación y Empleo

Fuente: [INEGI · ENOE](https://inegi.org.mx/programas/enoe/15ymas/#datos_abiertos)  
Formato: 85 archivos ZIP trimestrales (2005-T1 – 2026-T2)  
Módulos: **SDEM** (sociodemográfico + ingresos) y **COE1** (ocupación)  
Join: inner join por 13 llaves compuestas de identificación de persona  
Salida: Parquet bronce particionado por `anio`

> **Nota:** 2020-T2 no existe en la descarga masiva del INEGI (pandemia). No es un error.

In [22]:
# ── 4.1 Columnas a extraer y llaves de join ────────────────────────────────
import re, zipfile
from glob import glob
from functools import reduce

JOIN_KEYS = [
    "r_def", "cd_a", "ent", "con", "upm", "d_sem",
    "n_pro_viv", "v_sel", "n_hog", "h_mud", "n_ent", "per", "n_ren"
]

SDEM_COLS = JOIN_KEYS + [
    "sex", "eda", "anios_esc", "cs_p13_1", "niv_ins",
    "clase1", "clase2", "pos_ocu", "seg_soc",
    "ing7c", "ingocup", "hrsocup",
    "rama", "scian", "ur", "zona", "salario",
    "fac", "sub_o", "tip_con"
]

COE1_COLS = JOIN_KEYS + [
    "p1", "p3", "p3b", "p4", "p4b", "p4c",
    "p5c_thrs", "p5h"
]

print(f"SDEM columnas a extraer : {len(SDEM_COLS)}")
print(f"COE1 columnas a extraer : {len(COE1_COLS)}")
print(f"Llaves de join          : {len(JOIN_KEYS)}")

SDEM columnas a extraer : 33
COE1 columnas a extraer : 21
Llaves de join          : 13


In [23]:
# ── 4.2 Funciones auxiliares ────────────────────────────────────────────────

def parse_zip_name(fname):
    """Extrae (año, trimestre) del nombre del ZIP con tres patrones conocidos."""
    # Patron principal: *_YYYY_Nt_csv.zip  (2005-2016, 2018-2026)
    m = re.search(r"_(\d{4})_(\d)t_csv", fname)
    if m:
        return int(m.group(1)), int(m.group(2))
    # Patron alternativo: *_YYYY_trimN_*.zip  (2017)
    m = re.search(r"_(\d{4})_trim(\d)_", fname)
    if m:
        return int(m.group(1)), int(m.group(2))
    # Patron especial: *enoeYYYY_Nt_csv.zip  (2018-T2 con prefijo numérico)
    m = re.search(r"enoe(\d{4})_(\d)t_csv", fname)
    if m:
        return int(m.group(1)), int(m.group(2))
    return None, None


def fix_df(df, year, trim):
    """Normaliza BOM, renombra cve_ent->ent y fac_tri->fac; agrega anio/trimestre."""
    # Fix BOM en primera columna (archivos 2017 con UTF-8-SIG)
    fc = df.columns[0]
    if fc.startswith("\ufeff"):
        df = df.withColumnRenamed(fc, fc.replace("\ufeff", ""))
    # Renombrar cve_ent -> ent  (esquema 2026+)
    if "cve_ent" in df.columns and "ent" not in df.columns:
        df = df.withColumnRenamed("cve_ent", "ent")
    # Renombrar fac_tri -> fac  (esquema 2026+)
    if "fac_tri" in df.columns and "fac" not in df.columns:
        df = df.withColumnRenamed("fac_tri", "fac")
    return (
        df.withColumn("anio",      F.lit(year).cast(IntegerType()))
          .withColumn("trimestre", F.lit(trim).cast(IntegerType()))
    )


def safe_select(df, cols):
    """Selecciona columnas; rellena con NULL las que no existan en el CSV."""
    present = set(df.columns)
    return df.select([
        F.col(c) if c in present
        else F.lit(None).cast("string").alias(c)
        for c in cols
    ])

print("Funciones auxiliares definidas OK")

Funciones auxiliares definidas OK


In [24]:
# ── 4.3 Descomprimir, procesar y guardar Parquet por trimestre ─────────────
# Estrategia: escribir cada trimestre a Parquet (append) y borrar
# los CSVs descomprimidos de inmediato para ahorrar espacio en disco.
# Los ZIPs originales NO se tocan.
import shutil

zip_files = sorted(
    f for f in os.listdir(ENOE_DIR) if f.endswith(".zip")
)
print(f"ZIPs encontrados: {len(zip_files)}")

procesados = 0
errores    = []

for fname in zip_files:
    year, trim = parse_zip_name(fname)
    if year is None:
        errores.append(f"  Sin parsear: {fname}")
        continue

    out_dir = os.path.join(ENOE_RAW_DIR, f"{year}_{trim}")
    Path(out_dir).mkdir(parents=True, exist_ok=True)

    # 1. Descomprimir
    with zipfile.ZipFile(os.path.join(ENOE_DIR, fname)) as z:
        z.extractall(out_dir)

    # 2. Localizar CSVs
    # Cada ZIP trae tambien bitacora, diccionario y catalogos con "coe1"/"sdem"
    # en el nombre; nos quedamos solo con el CSV de datos (1 por modulo).
    EXCLUIR = ("bitacora", "diccionario", "catalogos", "metadatos")
    def buscar_csv(modulo):
        return [f for f in glob(f"{out_dir}/**/*{modulo}*.csv", recursive=True)
                if not any(x in f.lower() for x in EXCLUIR)]
    coe1_files = buscar_csv("coe1")
    sdem_files = buscar_csv("sdem")

    if not coe1_files or not sdem_files:
        errores.append(f"  CSV no encontrado: {year}-T{trim}")
        shutil.rmtree(out_dir)  # limpiar aunque no se use
        continue

    # 3. Leer con Spark
    opts = {"header": "true", "encoding": "latin1", "inferSchema": "true"}
    coe1 = fix_df(spark.read.options(**opts).csv(coe1_files[0]), year, trim)
    sdem = fix_df(spark.read.options(**opts).csv(sdem_files[0]), year, trim)

    # anio/trimestre solo en SDEM: si van en ambos lados, el join los duplica (AMBIGUOUS_REFERENCE)
    coe1_s = safe_select(coe1, COE1_COLS)
    sdem_s = safe_select(sdem, SDEM_COLS + ["anio", "trimestre"])

    joined = sdem_s.join(coe1_s, JOIN_KEYS, "inner")

    # inferSchema decide el tipo trimestre a trimestre (p.ej. eda: int en un
    # trimestre, string en otro) y Spark no puede leer la carpeta mezclada.
    # Bronce guarda todo como string; el casting real se hace en Plata.
    joined = joined.select([
        F.col(c) if c in ("anio", "trimestre") else F.col(c).cast("string").alias(c)
        for c in joined.columns
    ])

    # 4. Escribir Parquet (append) — fuerza ejecucion Spark antes de borrar
    (
        joined
        .write
        .mode("append")
        .partitionBy("anio")
        .parquet(ENOE_PARQUET)
    )

    # 5. Borrar CSVs descomprimidos (ZIP original intacto)
    shutil.rmtree(out_dir)
    procesados += 1
    print(f"  OK {year}-T{trim} | borrado {out_dir}")

print(f"\nTrimestres procesados : {procesados}")
if errores:
    print("Advertencias:")
    for e in errores: print(e)

ZIPs encontrados: 85
  OK 2007-T2 | borrado ..\data\raw\enoe\raw\2007_2
  OK 2007-T3 | borrado ..\data\raw\enoe\raw\2007_3
  OK 2007-T4 | borrado ..\data\raw\enoe\raw\2007_4
  OK 2008-T1 | borrado ..\data\raw\enoe\raw\2008_1
  OK 2008-T2 | borrado ..\data\raw\enoe\raw\2008_2
  OK 2008-T3 | borrado ..\data\raw\enoe\raw\2008_3
  OK 2008-T4 | borrado ..\data\raw\enoe\raw\2008_4
  OK 2009-T1 | borrado ..\data\raw\enoe\raw\2009_1
  OK 2009-T2 | borrado ..\data\raw\enoe\raw\2009_2
  OK 2009-T3 | borrado ..\data\raw\enoe\raw\2009_3
  OK 2005-T1 | borrado ..\data\raw\enoe\raw\2005_1
  OK 2009-T4 | borrado ..\data\raw\enoe\raw\2009_4
  OK 2010-T1 | borrado ..\data\raw\enoe\raw\2010_1
  OK 2010-T2 | borrado ..\data\raw\enoe\raw\2010_2
  OK 2010-T3 | borrado ..\data\raw\enoe\raw\2010_3
  OK 2010-T4 | borrado ..\data\raw\enoe\raw\2010_4
  OK 2011-T1 | borrado ..\data\raw\enoe\raw\2011_1
  OK 2011-T2 | borrado ..\data\raw\enoe\raw\2011_2
  OK 2011-T3 | borrado ..\data\raw\enoe\raw\2011_3
  OK 2011-

In [25]:
# ── 4.4 Verificar cobertura por ano y trimestre (desde Parquet) ─────────────
# Se lee de vuelta desde Parquet para confirmar que todo quedó escrito.
df_enoe = spark.read.parquet(ENOE_PARQUET)

print(f"Total filas     : {df_enoe.count():,}")
print(f"Total columnas  : {len(df_enoe.columns)}")

(
    df_enoe
    .groupBy("anio", "trimestre")
    .agg(F.count("*").alias("filas"))
    .orderBy("anio", "trimestre")
    .show(100, truncate=False)
)

Total filas     : 26,407,224
Total columnas  : 43
+----+---------+------+
|anio|trimestre|filas |
+----+---------+------+
|2005|1        |310935|
|2005|2        |313071|
|2005|3        |315174|
|2005|4        |318255|
|2006|1        |320829|
|2006|2        |319332|
|2006|3        |318985|
|2006|4        |318465|
|2007|1        |320688|
|2007|2        |319328|
|2007|3        |317097|
|2007|4        |311334|
|2008|1        |317188|
|2008|2        |315876|
|2008|3        |311863|
|2008|4        |310561|
|2009|1        |311619|
|2009|2        |310460|
|2009|3        |308313|
|2009|4        |310206|
|2010|1        |312167|
|2010|2        |313483|
|2010|3        |311211|
|2010|4        |308588|
|2011|1        |309470|
|2011|2        |308628|
|2011|3        |308151|
|2011|4        |308227|
|2012|1        |310372|
|2012|2        |310131|
|2012|3        |308136|
|2012|4        |304657|
|2013|1        |305122|
|2013|2        |304817|
|2013|3        |305084|
|2013|4        |309734|
|2014|1       

In [26]:
# ── 4.5 Schema y tamano final ───────────────────────────────────────────────
df_enoe.printSchema()

size_mb = sum(
    f.stat().st_size for f in Path(ENOE_PARQUET).rglob("*.parquet")
) / 1_048_576
print(f"Tamano Parquet ENOE : {size_mb:.1f} MB")
print(f"Ruta                : {ENOE_PARQUET}")

root
 |-- r_def: string (nullable = true)
 |-- cd_a: string (nullable = true)
 |-- ent: string (nullable = true)
 |-- con: string (nullable = true)
 |-- upm: string (nullable = true)
 |-- d_sem: string (nullable = true)
 |-- n_pro_viv: string (nullable = true)
 |-- v_sel: string (nullable = true)
 |-- n_hog: string (nullable = true)
 |-- h_mud: string (nullable = true)
 |-- n_ent: string (nullable = true)
 |-- per: string (nullable = true)
 |-- n_ren: string (nullable = true)
 |-- sex: string (nullable = true)
 |-- eda: string (nullable = true)
 |-- anios_esc: string (nullable = true)
 |-- cs_p13_1: string (nullable = true)
 |-- niv_ins: string (nullable = true)
 |-- clase1: string (nullable = true)
 |-- clase2: string (nullable = true)
 |-- pos_ocu: string (nullable = true)
 |-- seg_soc: string (nullable = true)
 |-- ing7c: string (nullable = true)
 |-- ingocup: string (nullable = true)
 |-- hrsocup: string (nullable = true)
 |-- rama: string (nullable = true)
 |-- scian: string (null

---
## 5 · Resumen de ingesta

Verificación final de las cuatro fuentes procesadas en este notebook.

In [27]:
# ── 5.1 Resumen final de las 4 fuentes ─────────────────────────────────────
print("=" * 60)
print("RESUMEN DE INGESTA — BRONCE")
print("=" * 60)

fuentes = [
    ("INPC", INPC_PARQUET, "anio"),
    ("SHCP", SHCP_PARQUET, "CICLO"),
    ("IGAE", IGAE_PARQUET, "CICLO"),
    ("ENOE", ENOE_PARQUET, "anio"),
]

for nombre, parquet_path, col_anio in fuentes:
    try:
        df = spark.read.parquet(parquet_path)
        rango = df.select(
            F.min(col_anio).alias("min"),
            F.max(col_anio).alias("max")
        ).collect()[0]
        size_mb = sum(
            f.stat().st_size for f in Path(parquet_path).rglob("*.parquet")
        ) / 1_048_576
        print(f"\n{nombre}")
        print(f"  Filas     : {df.count():,}")
        print(f"  Columnas  : {len(df.columns)}")
        print(f"  Periodo   : {rango['min']}-{rango['max']}")
        print(f"  Tamano    : {size_mb:.1f} MB (Parquet snappy)")
        print(f"  Ruta      : {parquet_path}")
    except Exception as e:
        print(f"\n{nombre} -- sin datos ({e})")

print("\n" + "=" * 60)
print("Siguiente paso: 05_limpieza_enoe.ipynb")
#spark.stop()

RESUMEN DE INGESTA — BRONCE

INPC
  Filas     : 1,988
  Columnas  : 6
  Periodo   : 2003-2026
  Tamano    : 0.1 MB (Parquet snappy)
  Ruta      : ..\data\raw\inpc\parquet

SHCP
  Filas     : 878
  Columnas  : 11
  Periodo   : 1990-2026
  Tamano    : 0.1 MB (Parquet snappy)
  Ruta      : ..\data\raw\shcp\parquet

IGAE
  Filas     : 1,468
  Columnas  : 6
  Periodo   : 1993-2026
  Tamano    : 0.1 MB (Parquet snappy)
  Ruta      : ..\data\raw\igae\parquet

ENOE
  Filas     : 26,407,224
  Columnas  : 43
  Periodo   : 2005-2026
  Tamano    : 451.3 MB (Parquet snappy)
  Ruta      : ..\data\raw\enoe\parquet

Siguiente paso: 05_limpieza_enoe.ipynb
